# Project XYZ — Track B: Experiment V2 (16K Streaming SFT for Coding & Capability Preservation)

This notebook executes **Experiment V2** on **`Qwen/Qwen3-4B-Instruct-2507`** using **4-bit NF4 QLoRA** on an **NVIDIA T4 GPU (16 GB VRAM)** in Google Colab.

### Key Experiment V2 Upgrades Over V1:
1. **True Progressive Streaming**: Consumes 16,000 unique examples dynamically from Hugging Face with `streaming=True` without downloading full datasets.
2. **Targeted Mixture**:
   - **45% OpenCoder** (`educational_instruct`): ~7,200 educational coding instruction pairs.
   - **25% BigCode** (`self-oss-instruct-sc2-exec-filter-50k`): ~4,000 execution-validated coding tasks.
   - **20% SmolTalk** (`smol-magpie-ultra`): ~3,200 general instruction and reasoning samples.
   - **10% Cybersecurity** (`Trendyol` + curated seeds): ~1,600 unique defensive security dialogues.
3. **Longer Context**: Full `MAX_SEQ_LENGTH = 2048` with assistant-only loss masking.
4. **Refined Hyperparameters**: LoRA rank $r=16, \alpha=32$, learning rate $5 \times 10^{-5}$, `max_steps=1000` (effective batch size 16 = 16,000 example exposures).
5. **Execution-Based Coding Benchmark**: Evaluates code correctness via a sandboxed unit test runner on 60 Python programming tasks.
6. **Google Drive Persistence**: Checkpoints stream to `MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2/`.

In [ ]:
# CELL 2: GPU Check and CUDA Verification
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable! In VS Code or Colab, connect to an NVIDIA T4 GPU runtime "
        "(Runtime -> Change runtime type -> T4 GPU)."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Connected to GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")

!nvidia-smi

In [ ]:
# CELL 3: Mount and Verify Google Drive (Experiment V2 Directory)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"Google Drive mount skipped (or running locally): {e}")
    DRIVE_MOUNTED = False

DRIVE_ROOT = "/content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2"
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/adapters", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/benchmarks", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)

print(f"Experiment V2 Google Drive root verified at: {DRIVE_ROOT}")

In [ ]:
# CELL 4: Clone / Access Repository on Branch 'qwen3-4b-stream-coding-v2'
import os, sys
from pathlib import Path

REPO_NAME = "project-xyz"
LOCAL_REPO = Path(f"/content/{REPO_NAME}")

if not LOCAL_REPO.exists() and not Path("src").exists():
    print("Cloning repository branch qwen3-4b-stream-coding-v2 into Colab runtime...")
    !git clone -b qwen3-4b-stream-coding-v2 https://github.com/sajancodes/project-xyz.git /content/project-xyz

if LOCAL_REPO.exists():
    os.chdir(str(LOCAL_REPO))

repo_path = str(Path.cwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Active working directory:", os.getcwd())
assert Path("src").exists(), "src directory not found!"

In [ ]:
# CELL 5: Install Pinned QLoRA Dependencies
!pip install --upgrade pip
!pip install torch>=2.2.0 transformers>=4.48.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 peft>=0.14.0 trl>=0.12.0 datasets>=2.21.0 pyyaml scikit-learn

In [ ]:
# CELL 6: Print Dependency Versions
import sys, torch, transformers, peft, trl, accelerate, bitsandbytes

print("=" * 50)
print("INSTALLED FRAMEWORK VERSIONS")
print("=" * 50)
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"Accelerate:   {accelerate.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")
print("=" * 50)

In [ ]:
# CELL 7: Experiment V2 Hyperparameters
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 2048        # Full context window for complex code & explanations
LORA_R = 16                  # Conservative rank for 16K dataset
LORA_ALPHA = 32              # 2 * rank
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5         # Lower, stable SFT learning rate
MICRO_BATCH_SIZE = 1         # T4 memory safety
GRADIENT_ACCUMULATION = 16   # Effective batch size = 16
MAX_STEPS = 1000             # 1000 * 16 = 16,000 example exposures
WARMUP_RATIO = 0.05
RESUME_FROM_CHECKPOINT = None # Set to explicit checkpoint path or None for auto-detect
DRIVE_ROOT = "/content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2"

print(f"Hyperparameters configured for 16K Streaming SFT on: {BASE_MODEL}")

In [ ]:
# CELL 8: Materialize Finite Validation and Test Datasets from Streams
from src.streaming.sampler import build_finite_eval_datasets

print("Building finite validation (600) and test (400) datasets from streams...")
val_path, test_path = build_finite_eval_datasets(
    output_dir="data/qwen4b/processed",
    val_target=600,
    test_target=400,
)
print(f"Finite evaluation datasets ready:")
print(f"  - Validation: {val_path}")
print(f"  - Test:       {test_path}")

In [ ]:
# CELL 9: Load Qwen Tokenizer
from src.models.qwen4b.loader import load_qwen_tokenizer

tokenizer = load_qwen_tokenizer(BASE_MODEL)
print(f"Official Qwen tokenizer loaded. Vocab size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}', EOS token: '{tokenizer.eos_token}'")

In [ ]:
# CELL 10: Load Qwen 4B in NF4 4-bit
from src.models.qwen4b.loader import load_qwen_model

print(f"Loading {BASE_MODEL} in 4-bit NF4 (fp16 compute dtype)...\n")
model = load_qwen_model(
    model_id_or_path=BASE_MODEL,
    load_in_4bit=True,
    compute_dtype="float16",
    device_map="auto",
    use_cache=False,
    enable_gradient_checkpointing=True,
)
print("Base model successfully loaded and quantized at runtime!")

In [ ]:
# CELL 11: Attach LoRA Adapter (r=16, alpha=32)
from src.models.qwen4b.lora import apply_qlora, get_qlora_config

lora_config = get_qlora_config(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = apply_qlora(model, lora_config=lora_config, prepare_kbit=True)
print("LoRA adapter attached successfully!")

In [ ]:
# CELL 12: Inspect Model Parameters
from src.models.qwen4b.loader import get_model_parameters_info
from src.training.qwen4b.utils import print_gpu_environment

print_gpu_environment()
param_info = get_model_parameters_info(model)
print(f"Total Parameters:     {param_info['total_parameters']:,}")
print(f"Trainable Parameters: {param_info['trainable_parameters']:,}")
print(f"Trainable Ratio:      {param_info['trainable_percentage']:.4f}%")

In [ ]:
# CELL 13: Forward / Backward Smoke Test (T4 OOM Verification)
from src.training.qwen4b.utils import run_smoke_forward_backward_step

success, msg = run_smoke_forward_backward_step(model, tokenizer, max_seq_len=256)
if not success:
    raise RuntimeError(f"Smoke test failed! {msg}")
print(msg)

In [ ]:
# CELL 14: Run BASELINE Benchmarks (Multi-Domain + 60 Execution-Based Coding Tasks)
from src.evaluation.coding.runner import evaluate_coding_benchmark
from src.evaluation.qwen4b.benchmark import run_benchmark

print("1. Running Baseline Multi-Domain Benchmark (Cybersecurity, Reasoning, General)...")
base_multi_results = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    tag="base_v2",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)
print(f"Baseline Multi-Domain Score: {base_multi_results['overall_score']:.4f}")

print("\n2. Running Baseline Execution-Based Coding Benchmark (60 Tasks)...\n")
base_coding_results = evaluate_coding_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_path=f"{DRIVE_ROOT}/benchmarks/base_coding_benchmark.json",
)
print(f"Baseline Coding Pass@1: {base_coding_results['pass_at_1']}% ({base_coding_results['passed_count']}/{base_coding_results['num_tasks']})")

In [ ]:
# CELL 15: Initialize Streaming Trainer and Progressive Datasets
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments
from src.streaming.streamer import StreamSFTDataset
from src.training.qwen4b.callbacks import LiveTrainingProgressCallback
from src.training.qwen4b.checkpointing import DriveBackupCallback, GoogleDriveCheckpointManager
from src.training.qwen4b.collator import DataCollatorForAssistantOnlyLM
from src.training.qwen4b.dataset import CyberChatDataset

drive_mgr = GoogleDriveCheckpointManager(DRIVE_ROOT)
drive_mgr.ensure_directories()

train_stream = StreamSFTDataset(
    tokenizer=tokenizer,
    partition="train",
    max_seq_length=MAX_SEQ_LENGTH,
    total_target=16000,
    quotas={"opencoder": 7200, "bigcode": 4000, "smoltalk": 3200, "cybersecurity": 1600},
    shuffle_buffer_size=10000,
    seed=42,
)

val_dataset = CyberChatDataset(val_path, tokenizer=tokenizer, max_length=MAX_SEQ_LENGTH)
collator = DataCollatorForAssistantOnlyLM(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="outputs/qwen3_4b_coding_stream_v2",
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
)

callbacks = [
    DriveBackupCallback(drive_mgr),
    LiveTrainingProgressCallback(max_steps=MAX_STEPS, log_every_n_steps=10, tracker=train_stream.tracker),
    EarlyStoppingCallback(early_stopping_patience=3),
]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_stream,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=callbacks,
)

resume_target = RESUME_FROM_CHECKPOINT or drive_mgr.get_resume_checkpoint(Path("outputs/qwen3_4b_coding_stream_v2"))
print(f"Trainer initialized for streaming! Resume checkpoint target: {resume_target}")

In [ ]:
# ====================================================================
# ====================================================================
#            CELL 16: FINAL STREAMING TRAINING CELL                  
# ====================================================================
# ====================================================================
# Execute this cell to begin full 16,000-example Streaming SFT (max_steps=1000).
# Checkpoints automatically persist to Google Drive!

print("Starting Experiment V2 Streaming Training on NVIDIA T4...")
train_output = trainer.train(resume_from_checkpoint=resume_target)

print("=" * 60)
print("STREAMING QLORA TRAINING COMPLETED!")
print("=" * 60)
print(train_output)

# Print Source Intake Statistics
print("\n" + train_stream.tracker.get_summary_table())

In [ ]:
# CELL 17: Save Final LoRA Adapter Locally and to Google Drive
local_adapter = Path("outputs/qwen3_4b_coding_stream_v2/final_adapter")
local_adapter.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(local_adapter))
tokenizer.save_pretrained(str(local_adapter))

drive_mgr.sync_adapter_to_drive(local_adapter, adapter_name="final_stream_coding_adapter")
print(f"Saved final adapter locally to: {local_adapter}")
print(f"Persisted adapter to Drive: {DRIVE_ROOT}/adapters/final_stream_coding_adapter")

In [ ]:
# CELL 18: Clean Training State from GPU Memory
from src.training.qwen4b.utils import clear_cuda_memory

del trainer
del model
clear_cuda_memory()
print("Training state cleared from GPU memory.")

In [ ]:
# CELL 19: Reload Best Adapter from Google Drive
from src.models.qwen4b.inference import load_lora_adapter
from src.models.qwen4b.loader import load_qwen_model

print("Reloading base model in NF4 4-bit...")
eval_base_model = load_qwen_model(BASE_MODEL, load_in_4bit=True, device_map="auto")

adapter_path = f"{DRIVE_ROOT}/adapters/final_stream_coding_adapter"
print(f"Attaching fine-tuned streaming adapter: {adapter_path}...")
eval_model = load_lora_adapter(eval_base_model, adapter_path)
print("Adapter reloaded successfully!")

In [ ]:
# CELL 20: Post-Training Benchmarks
from src.evaluation.coding.runner import evaluate_coding_benchmark
from src.evaluation.qwen4b.benchmark import run_benchmark

print("1. Running Post-Training Multi-Domain Benchmark...")
post_multi_results = run_benchmark(
    model=eval_model,
    tokenizer=tokenizer,
    tag="qlora_v2_stream",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)
print(f"Post-Training Multi-Domain Score: {post_multi_results['overall_score']:.4f}")

print("\n2. Running Post-Training Execution-Based Coding Benchmark (60 Tasks)...\n")
post_coding_results = evaluate_coding_benchmark(
    model=eval_model,
    tokenizer=tokenizer,
    output_path=f"{DRIVE_ROOT}/benchmarks/post_coding_benchmark.json",
)
print(f"Post-Training Coding Pass@1: {post_coding_results['pass_at_1']}% ({post_coding_results['passed_count']}/{post_coding_results['num_tasks']})")

In [ ]:
# CELL 21: Generate Before vs After Comparison Report
from IPython.display import Markdown, display
from src.evaluation.qwen4b.comparison import compare_benchmarks

report_md = compare_benchmarks(
    base_results=base_multi_results,
    qlora_results=post_multi_results,
    output_md_path=f"{DRIVE_ROOT}/reports/comparison_report_v2.md",
)

coding_summary = (
    f"\n### Execution-Based Coding Benchmark (60 Tasks)\n\n"
    f"| Model | Tasks Passed | Total Tasks | Pass@1 Accuracy |\n"
    f"|---|---:|---:|---:|\n"
    f"| Base Qwen 4B | {base_coding_results['passed_count']} | {base_coding_results['num_tasks']} | {base_coding_results['pass_at_1']}% |\n"
    f"| Streaming QLoRA v2 | {post_coding_results['passed_count']} | {post_coding_results['num_tasks']} | {post_coding_results['pass_at_1']}% |\n"
    f"| **Delta** | **+{post_coding_results['passed_count'] - base_coding_results['passed_count']}** | -- | **+{post_coding_results['pass_at_1'] - base_coding_results['pass_at_1']:.2f}%** |\n"
)

display(Markdown(report_md + coding_summary))

In [ ]:
# CELL 22: Interactive Code & Cybersecurity Query Testing
from src.models.qwen4b.inference import generate_response

prompt = "Write a Python function to find the longest palindromic substring in a string, with full unit tests." # @param {type:"string"}
response = generate_response(eval_model, tokenizer, prompt, max_new_tokens=512)
print("Prompt:", prompt)
print("\nResponse:\n", response)